# Chat History Memory


[Step 11 - Memory and session state]

> **MLCourse - Agentic AI - Memory and State**

> Stage in the capstone: the capstone chatbot remembers prior turns PER THREAD thanks to this.

Chat models are stateless: every call starts from zero, so nothing survives
unless YOU store it and re-send it. This notebook builds memory from first
principles, then hands the plumbing to **LangGraph persistence** - a graph plus
a *checkpointer* that saves and reloads state under a `thread_id`.

> **Why LangGraph and not `RunnableWithMessageHistory`?** The older wrapper is
> deprecated. LangGraph's checkpointer is the supported replacement and it is
> strictly more capable: it persists your whole state (not just a message list),
> it powers time-travel and human-in-the-loop, and the same checkpointer object
> swaps between in-memory and SQLite without touching your chain.

# What you will learn

1. Part A - why memory is needed: watch a message list grow turn by turn (token bloat).
2. Part B - a minimal LangGraph app + `InMemorySaver` keyed by `thread_id`,
   with PROOF that two threads never share memories.
3. Part C - window memory: trim to the last N messages, measure the char savings,
   and observe exactly WHAT a windowed model forgets.
4. Part D - persistence: swap `InMemorySaver` for `SqliteSaver` and reload a
   conversation from disk after the "process" is gone.

### Sections

1. Setup
2. Part A - the growth problem, measured in characters
3. Part B - graph + checkpointer + thread isolation proof
4. Part C - sliding window: cheaper, forgetful on purpose
5. Part D - SQLite persistence round trip
6. Summary

### Section 1: setup


In [ ]:
from pathlib import Path          # cross-platform paths
import os                         # environment access for API keys
import re                         # regex used by the offline stub to find names

def _find_track(start_dir):
    """Climb parent folders until we find (or reach) the dir named 03_agentic_ai."""
    here = Path(start_dir).resolve()
    for candidate in (here, *here.parents):
        if candidate.name == "03_agentic_ai":
            return candidate
        if (candidate / "03_agentic_ai").is_dir():
            return candidate / "03_agentic_ai"
    raise FileNotFoundError("Could not locate the 03_agentic_ai track near %s" % here)

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"             # shared data dir (sqlite file will land here too)
DATA.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv    # load GROQ/HF keys if the learner configured any
load_dotenv(TRACK / ".env", override=False)
load_dotenv(override=False)

try:                              # Jupyter-only magic; harmless in plain Python
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("track:", TRACK)
print("data :", DATA)


### Section 2 (Part A): the message-list growth problem


In [ ]:
# A "memory" is just a Python list of messages you append to. The catch: most
# chat APIs make you RE-SEND the entire list every turn, so cost grows with every
# exchange. We simulate four turns and count characters (roughly tokens / 4).
from langchain_core.messages import HumanMessage, AIMessage

TURNS = [
    ("Hi, I am Thoya.",                            "Nice to meet you, Thoya!"),
    ("Remember: my favorite color is teal.",       "Teal it is - locked in."),
    ("What was my favorite color?",                "You told me: teal."),
    ("And my name?",                               "You introduced yourself as Thoya."),
]

conversation = []                  # THIS list is the entire "memory"
print("turn | messages | chars total   (each turn resends ALL of this)")
for i, (user_text, bot_text) in enumerate(TURNS, start=1):
    conversation.append(HumanMessage(content=user_text))   # remember what user said
    conversation.append(AIMessage(content=bot_text))       # remember what bot said
    total_chars = sum(len(m.content) for m in conversation)
    print(" %3d  |   %2d     | %6d" % (i, len(conversation), total_chars))

print("\nlesson: linear growth per turn, quadratic resend cost over a long chat;")
print("untrimmed buffers eventually overflow the model's context window.")


### Section 3a (Part B): guarded chat chain ready for memory


In [ ]:
# The prompt has three slots: fixed system rules, a 'history' placeholder the
# wrapper will fill, and the new human input. The model itself follows the track's
# provider rules: Ollama llama3.2 primary (probed once), offline stub fallback.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

MEMORY_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly assistant with a reliable memory. "
     "When asked for the user's name, answer with exactly the name they gave earlier."),
    MessagesPlaceholder(variable_name="history"),   # injected turns go HERE
    ("human", "{input}"),                           # the newest user message
])

from langchain_ollama import ChatOllama
base_llm = ChatOllama(model="llama3.2", temperature=0)

LLM_LIVE = False
try:
    base_llm.invoke("Reply with the single word: pong")     # reachability probe
    LLM_LIVE = True
except Exception as exc:
    print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
    print("   detail: %s: %s" % (type(exc).__name__, exc))

def fake_memory_reply(prompt_value):
    """OFFLINE STUB so the PIPELINE mechanics run anywhere; swap llm back for real answers.

    Reads ONLY what is visible inside the prompt: scans visible messages for
    'my name is X' and answers name questions deterministically. Because it sees
    just the injected history, it also faithfully demonstrates trimming effects.
    """
    try:
        msgs = prompt_value.to_messages()
    except Exception:
        msgs = []
    last_human = ""
    for m in reversed(msgs):
        if getattr(m, "type", "") == "human":
            last_human = m.content
            break
    if "name" in last_human.lower():
        blob = " ".join(getattr(m, "content", "") for m in msgs)
        found = re.findall(r"name is ([A-Za-z]+)", blob, flags=re.IGNORECASE)
        if found:
            return "Your name is %s." % found[-1]
        return "I do not see your name anywhere in our conversation."
    return "[offline stub] You said: \"%s\" (swap llm back for real answers)" % last_human[:100]

if LLM_LIVE:
    llm = base_llm
else:
    from langchain_core.runnables import RunnableLambda
    llm = RunnableLambda(fake_memory_reply)
    print(">> running with the OFFLINE STUB model for this session")

chat_chain = MEMORY_PROMPT | llm | StrOutputParser()   # pure chain: no storage yet


### Section 3b (Part B): the graph + a checkpointer


In [ ]:
# This is the modern replacement for RunnableWithMessageHistory. Three pieces:
#
#   1. STATE  - `MessagesState` is a ready-made TypedDict whose single `messages`
#               key uses an *append* reducer, so whatever a node returns is added
#               to the running list rather than overwriting it.
#   2. NODE   - one function that reads the accumulated messages, calls our chain,
#               and returns the new AI message.
#   3. SAVER  - a checkpointer. On every invoke LangGraph loads the state for the
#               given `thread_id`, runs the graph, and writes the new state back.
#
# The `thread_id` plays exactly the role `session_id` used to: it is the key that
# separates one conversation from another.
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage

def call_model(state: MessagesState) -> dict:
    """Split the accumulated messages into 'everything before' + 'the new turn'."""
    history = state["messages"][:-1]        # past turns feed the placeholder
    latest = state["messages"][-1].content  # the newest human message
    reply = chat_chain.invoke({"history": history, "input": latest})
    return {"messages": [AIMessage(content=reply)]}   # appended, not replaced

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")

memory_saver = InMemorySaver()               # the whole "storage layer"
chat_with_memory = builder.compile(checkpointer=memory_saver)

CFG_ABC = {"configurable": {"thread_id": "abc"}}
CFG_XYZ = {"configurable": {"thread_id": "xyz"}}

def say(thread_cfg, text, app=None):
    """Guarded one-turn helper: prints both sides, returns the reply string."""
    app = app if app is not None else chat_with_memory
    print("You >", text)
    result = app.invoke({"messages": [HumanMessage(content=text)]}, config=thread_cfg)
    reply = result["messages"][-1].content
    print("Bot >", str(reply)[:300])
    return str(reply)

# THE ISOLATION PROOF: identical question pattern, different threads.
say(CFG_ABC, "Hello! My name is Thoya.")
say(CFG_XYZ, "Hi there! My name is Ada.")
a = say(CFG_ABC, "What is my name?")      # must recall THOYA
b = say(CFG_XYZ, "What is my name?")      # must recall ADA
print("-" * 60)
if a and b and "Thoya" in a and "Ada" in b:
    print("isolation HOLDS: same question, two threads, two identities")
else:
    print("isolation FAILED - check that each config uses a distinct thread_id")


### Section 4 (Part C): window memory - cheaper and deliberately forgetful


In [ ]:
# A buffer keeps everything; a WINDOW keeps only the last N messages. We push two
# filler turns straight into the checkpointed state (no model calls) so the
# forgetting effect is deterministic and costs zero tokens.
#
# `update_state` is the LangGraph equivalent of reaching into the old history
# object and appending by hand - except it goes through the checkpointer, so the
# writes are persisted exactly like real turns.
MAX_WINDOW_MESSAGES = 4                   # keep at most the last 4 messages

def trim_to_window(messages, max_messages=MAX_WINDOW_MESSAGES):
    """Sliding-window trim: newest max_messages survive, older ones are dropped."""
    if len(messages) <= max_messages:
        return list(messages)
    return list(messages[-max_messages:])

for u, a_txt in [("Tell me something about clocks.",
                  "Clocks tick; the White Rabbit panics about them."),
                 ("Tell me something about maps.",
                  "Maps fold space onto paper; useful, rarely punctual.")]:
    chat_with_memory.update_state(
        CFG_ABC, {"messages": [HumanMessage(content=u), AIMessage(content=a_txt)]}
    )

# Read the persisted state back out of the checkpointer.
full_msgs = chat_with_memory.get_state(CFG_ABC).values["messages"]
win_msgs = trim_to_window(full_msgs)
chars_full = sum(len(m.content) for m in full_msgs)
chars_win = sum(len(m.content) for m in win_msgs)
saved = 100 * (chars_full - chars_win) // chars_full if chars_full else 0
print("buffer : %d messages, %d chars" % (len(full_msgs), chars_full))
print("window : %d messages, %d chars (%d%% smaller)" % (len(win_msgs), chars_win, saved))
for m in win_msgs:
    print("   kept [%s] %s" % (m.type, m.content[:60]))

# Same graph shape and the SAME checkpointer, but the node trims what it forwards
# to the prompt. Note the state on disk is untouched - only the model's view shrinks.
def call_model_windowed(state: MessagesState) -> dict:
    """Identical to call_model, except the history is trimmed to a window."""
    history = trim_to_window(state["messages"][:-1])
    latest = state["messages"][-1].content
    reply = chat_chain.invoke({"history": history, "input": latest})
    return {"messages": [AIMessage(content=reply)]}

win_builder = StateGraph(MessagesState)
win_builder.add_node("model", call_model_windowed)
win_builder.add_edge(START, "model")
chat_with_window = win_builder.compile(checkpointer=memory_saver)   # shared storage

# Order matters for this experiment! We ask the WINDOWED chain first, while the
# only recent messages are the clocks/maps filler. If we asked the full-buffer
# chain first, its answer would land in the transcript and slide right back into
# the window - and the demo would silently prove nothing.
print("-" * 60)
print("WINDOWED chain (sees only the last %d messages):" % MAX_WINDOW_MESSAGES)
say(CFG_ABC, "What is my name?", app=chat_with_window)
print("FULL buffer chain (sees every message ever stored):")
say(CFG_ABC, "What is my name?", app=chat_with_memory)
print("\nsee the trade-off? the name fell OUT of the window, so the windowed bot")
print("could not answer - while the full buffer still had the very first turn.")
print("production fix: put critical facts in a summary or system note, not hope.")


### Section 5 (Part D): SQLite persistence


In [ ]:
# InMemorySaver dies with the process. SqliteSaver writes checkpoints to a real
# database file, keyed by thread_id - same graph, same code, ONE swapped object.
# That is the whole point of the checkpointer abstraction.
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

SQLITE_PATH = DATA / "chat_checkpoints.sqlite"
if SQLITE_PATH.exists():
    SQLITE_PATH.unlink()                  # clean slate so reruns stay deterministic

CFG_SQL = {"configurable": {"thread_id": "sql-demo"}}

# --- "process 1": write a couple of turns, then close the connection ----------
conn = sqlite3.connect(str(SQLITE_PATH), check_same_thread=False)
durable_app = builder.compile(checkpointer=SqliteSaver(conn))
durable_app.update_state(CFG_SQL, {"messages": [
    HumanMessage(content="Please remember this across restarts."),
    AIMessage(content="Saved to SQLite on disk."),
]})
conn.close()                              # simulate the process going away
print("wrote checkpoint to", SQLITE_PATH.name, "then closed the connection")

# --- "process 2": reopen the same file and read the conversation back ---------
conn2 = sqlite3.connect(str(SQLITE_PATH), check_same_thread=False)
reloaded_app = builder.compile(checkpointer=SqliteSaver(conn2))
restored = reloaded_app.get_state(CFG_SQL).values["messages"]
print("reopened the sqlite file and read back %d message(s):" % len(restored))
for m in restored:
    print("   [%s] %s" % (m.type, m.content))

# Thread isolation survives the round trip to disk as well.
stranger = reloaded_app.get_state({"configurable": {"thread_id": "someone-else"}})
other_msgs = stranger.values.get("messages", []) if stranger.values else []
print("other thread sees %d message(s) - isolation persists on disk too"
      % len(other_msgs))
conn2.close()
print("to go to production: swap SqliteSaver for PostgresSaver, nothing else changes")


### Summary

- Memory = store past turns under a key, inject them before each call, append
  the new pair after. **LangGraph's checkpointer automates all three steps.**
- The thread contract is one line:
  `invoke({"messages": [...]}, config={"configurable": {"thread_id": "abc"}})`.
- `MessagesState` supplies the append reducer, so a node returns only the NEW
  message and LangGraph accumulates the transcript for you.
- Two threads, two names, two answers proved isolation; one shared dict would
  have leaked them into each other.
- Window memory trades perfect recall for constant cost - and drops whatever
  falls out, as our name-in-the-past experiment showed live. Trimming happens
  in the node, so the full transcript stays safe in the checkpointer.
- Swap `InMemorySaver` -> `SqliteSaver` -> `PostgresSaver` when sessions must
  survive process restarts. The graph code does not change at all.

> **Migrating older code?** `RunnableWithMessageHistory` is deprecated. The
> mapping is direct: `session_id` -> `thread_id`, the history factory ->
> a checkpointer, and `history_messages_key` -> whatever your node reads out
> of state.

Next: Step 12 wires THIS machinery around the Step 10 RAG core. That is the capstone.